In [63]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


import pandas as pd

#AI Manufacturing Shortage & Production Risk Agen
data = {
    "Part_Number": ["P-101", "P-102", "P-103", "P-104"],
    "Description": [
        "Heating Element",
        "Temperature Sensor",
        "Control Module",
        "Wiring Harness"
    ],
    "Required_Qty": [500, 500, 500, 500],
    "Inventory_Qty": [320, 475, 200, 550],
    "Open_PO_Qty": [100, 100, 150, 0],
    "Lead_Time_Days": [21, 7, 30, 14],
    "Unit_Cost": [42, 18, 85, 26]
}

# Convert the data into a table
df = pd.DataFrame(data)

# Display the table
df

In [64]:
import pandas as pd

In [65]:
data = {
    "Part_Number": ["P-101", "P-102", "P-103", "P-104"],
    "Description": ["Heating Element", "Temperature Sensor", "Control Module", "Wiring Harness"],
    "Required_Qty": [500, 500, 500, 500],
    "Inventory_Qty": [320, 475, 200, 550],
    "Open_PO_Qty": [100, 100, 150, 0],
    "Lead_Time_Days": [21, 7, 30, 14],
    "Unit_Cost": [42, 18, 85, 26]
}

df = pd.DataFrame(data)
df

,Part_Number,Description,Required_Qty,Inventory_Qty,Open_PO_Qty,Lead_Time_Days,Unit_Cost
0,P-101,Heating Element,500,320,100,21,42
1,P-102,Temperature Sensor,500,475,100,7,18
2,P-103,Control Module,500,200,150,30,85
3,P-104,Wiring Harness,500,550,0,14,26


In [66]:
# Calculate total material available
df["Net_Available"] = df["Inventory_Qty"] + df["Open_PO_Qty"]

# Calculate shortage quantity
df["Shortage_Qty"] = df["Required_Qty"] - df["Net_Available"]

# Prevent negative numbers when we have enough inventory
df["Shortage_Qty"] = df["Shortage_Qty"].clip(lower=0)

# Display the updated table
df

,Part_Number,Description,Required_Qty,Inventory_Qty,Open_PO_Qty,Lead_Time_Days,Unit_Cost,Net_Available,Shortage_Qty
0,P-101,Heating Element,500,320,100,21,42,420,80
1,P-102,Temperature Sensor,500,475,100,7,18,575,0
2,P-103,Control Module,500,200,150,30,85,350,150
3,P-104,Wiring Harness,500,550,0,14,26,550,0


In [67]:
# Create a function to determine production shortage risk
def calculate_risk(row):
    if row["Shortage_Qty"] == 0:
        return "OK"
    elif row["Shortage_Qty"] >= 100 and row["Lead_Time_Days"] >= 21:
        return "HIGH"
    elif row["Shortage_Qty"] > 0 and row["Lead_Time_Days"] >= 14:
        return "MEDIUM"
    else:
        return "LOW"

# Apply the risk logic to every part
df["Risk_Level"] = df.apply(calculate_risk, axis=1)

# Display the results
df[[
    "Part_Number",
    "Description",
    "Shortage_Qty",
    "Lead_Time_Days",
    "Risk_Level"
]]

,Part_Number,Description,Shortage_Qty,Lead_Time_Days,Risk_Level
0,P-101,Heating Element,80,21,MEDIUM
1,P-102,Temperature Sensor,0,7,OK
2,P-103,Control Module,150,30,HIGH
3,P-104,Wiring Harness,0,14,OK


In [68]:
# Create purchasing recommendations
def recommend_action(row):
    if row["Risk_Level"] == "HIGH":
        return (
            f"URGENT: {row['Part_Number']} {row['Description']} has a "
            f"{row['Shortage_Qty']}-unit shortage with a "
            f"{row['Lead_Time_Days']}-day supplier lead time. "
            "Recommend expediting the existing purchase order "
            "or evaluating an alternate supplier."
        )

    elif row["Risk_Level"] == "MEDIUM":
        return (
            f"WARNING: {row['Part_Number']} {row['Description']} has a "
            f"{row['Shortage_Qty']}-unit shortage. "
            "Review the open purchase order and confirm the supplier "
            "delivery date."
        )

    elif row["Risk_Level"] == "LOW":
        return (
            f"MONITOR: {row['Part_Number']} {row['Description']} has a "
            "small shortage. Continue monitoring inventory and demand."
        )

    else:
        return (
            f"OK: {row['Part_Number']} {row['Description']} has sufficient "
            "material available for current production demand."
        )

# Apply recommendation logic
df["Agent_Recommendation"] = df.apply(recommend_action, axis=1)

# Display recommendations
for recommendation in df["Agent_Recommendation"]:
    print(recommendation)
    print()


OK: P-102 Temperature Sensor has sufficient material available for current production demand.

URGENT: P-103 Control Module has a 150-unit shortage with a 30-day supplier lead time. Recommend expediting the existing purchase order or evaluating an alternate supplier.

OK: P-104 Wiring Harness has sufficient material available for current production demand.



In [69]:
# TEST SCENARIO:
# Supplier adds 200 units to the P-103 open purchase order

df.loc[df["Part_Number"] == "P-103", "Open_PO_Qty"] = 350

# Recalculate available inventory
df["Net_Available"] = df["Inventory_Qty"] + df["Open_PO_Qty"]

# Recalculate shortage
df["Shortage_Qty"] = (
    df["Required_Qty"] - df["Net_Available"]
).clip(lower=0)

# Recalculate risk
df["Risk_Level"] = df.apply(calculate_risk, axis=1)

# Generate new recommendations
df["Agent_Recommendation"] = df.apply(recommend_action, axis=1)

# Show P-103 test result
df.loc[
    df["Part_Number"] == "P-103",
    [
        "Part_Number",
        "Description",
        "Open_PO_Qty",
        "Net_Available",
        "Shortage_Qty",
        "Risk_Level",
        "Agent_Recommendation"
    ]
]

,Part_Number,Description,Open_PO_Qty,Net_Available,Shortage_Qty,Risk_Level,Agent_Recommendation
2,P-103,Control Module,350,550,0,OK,OK: P-103 Control Module has sufficient materi...


In [70]:
# Restore P-103 to the original test condition
df.loc[df["Part_Number"] == "P-103", "Open_PO_Qty"] = 150

# Add number of days until the part is needed for production
df["Days_Until_Production"] = [14, 20, 12, 30]

# Recalculate inventory position
df["Net_Available"] = df["Inventory_Qty"] + df["Open_PO_Qty"]

df["Shortage_Qty"] = (
    df["Required_Qty"] - df["Net_Available"]
).clip(lower=0)

# Calculate whether supplier lead time exceeds production window
df["Days_Late"] = (
    df["Lead_Time_Days"] - df["Days_Until_Production"]
).clip(lower=0)

# Display production timing risk
df[[
    "Part_Number",
    "Description",
    "Shortage_Qty",
    "Lead_Time_Days",
    "Days_Until_Production",
    "Days_Late"
]]

,Part_Number,Description,Shortage_Qty,Lead_Time_Days,Days_Until_Production,Days_Late
0,P-101,Heating Element,80,21,14,7
1,P-102,Temperature Sensor,0,7,20,0
2,P-103,Control Module,150,30,12,18
3,P-104,Wiring Harness,0,14,30,0


In [71]:
# Enhanced risk assessment using shortage AND production timing
def calculate_production_risk(row):

    if row["Shortage_Qty"] == 0:
        return "OK"

    elif row["Days_Late"] >= 14:
        return "CRITICAL"

    elif row["Days_Late"] >= 7:
        return "HIGH"

    elif row["Days_Late"] > 0:
        return "MEDIUM"

    else:
        return "LOW"


# Apply enhanced production risk logic
df["Production_Risk"] = df.apply(
    calculate_production_risk,
    axis=1
)

# Display results
df[[
    "Part_Number",
    "Description",
    "Shortage_Qty",
    "Lead_Time_Days",
    "Days_Until_Production",
    "Days_Late",
    "Production_Risk"
]]

,Part_Number,Description,Shortage_Qty,Lead_Time_Days,Days_Until_Production,Days_Late,Production_Risk
0,P-101,Heating Element,80,21,14,7,HIGH
1,P-102,Temperature Sensor,0,7,20,0,OK
2,P-103,Control Module,150,30,12,18,CRITICAL
3,P-104,Wiring Harness,0,14,30,0,OK


In [72]:
# Generate production-risk recommendations
def production_recommendation(row):

    if row["Production_Risk"] == "CRITICAL":
        return (
            f"CRITICAL: {row['Part_Number']} {row['Description']} is "
            f"{row['Shortage_Qty']} units short. Production requires the "
            f"component in {row['Days_Until_Production']} days, but supplier "
            f"lead time is {row['Lead_Time_Days']} days, creating a "
            f"{row['Days_Late']}-day timing gap. "
            "Recommend immediately evaluating PO expediting, alternate "
            "suppliers, or production schedule adjustments."
        )

    elif row["Production_Risk"] == "HIGH":
        return (
            f"HIGH: {row['Part_Number']} {row['Description']} is "
            f"{row['Shortage_Qty']} units short with a "
            f"{row['Days_Late']}-day timing gap. "
            "Recommend contacting the supplier to confirm delivery and "
            "evaluate expedited shipping."
        )

    elif row["Production_Risk"] == "MEDIUM":
        return (
            f"MEDIUM: {row['Part_Number']} {row['Description']} has a "
            "potential production timing risk. Monitor supplier delivery "
            "and inventory status."
        )

    elif row["Production_Risk"] == "LOW":
        return (
            f"LOW: {row['Part_Number']} {row['Description']} has a shortage, "
            "but current supplier timing can support the production window."
        )

    else:
        return (
            f"OK: {row['Part_Number']} {row['Description']} currently has "
            "sufficient material for production."
        )


# Apply recommendation engine
df["Production_Recommendation"] = df.apply(
    production_recommendation,
    axis=1
)

# Print the agent's recommendations
for recommendation in df["Production_Recommendation"]:
    print(recommendation)
    print()

HIGH: P-101 Heating Element is 80 units short with a 7-day timing gap. Recommend contacting the supplier to confirm delivery and evaluate expedited shipping.

OK: P-102 Temperature Sensor currently has sufficient material for production.

CRITICAL: P-103 Control Module is 150 units short. Production requires the component in 12 days, but supplier lead time is 30 days, creating a 18-day timing gap. Recommend immediately evaluating PO expediting, alternate suppliers, or production schedule adjustments.

OK: P-104 Wiring Harness currently has sufficient material for production.



In [73]:
# AI Manufacturing Shortage & Production Risk Agent

### AI-Assisted Manufacturing Decision-Support Prototype

**Version 1.0**

## Project Overview

This project demonstrates an AI-assisted, rule-based manufacturing decision-support prototype developed using Python, Pandas, Jupyter Notebook, and AI-assisted vibe coding.

The prototype analyzes manufacturing inventory, open purchase orders, production requirements, supplier lead times, and production timing to identify potential component shortages before they disrupt production.

## What the Prototype Does

- Calculates available inventory
- Identifies material shortages
- Evaluates supplier lead-time risk
- Compares supplier lead time with production requirements
- Classifies production risk
- Generates recommended purchasing and production actions
- Maintains human approval for purchasing and supplier decisions

## Business Problem

Manufacturing organizations can experience production delays when component shortages are identified too late.

The objective of this prototype is to provide production planners and purchasing professionals with earlier visibility into potential material shortages and supplier timing risks.

## Decision Workflow

**Production Demand → Inventory → Open Purchase Orders → Shortage Detection → Supplier Lead Time → Production Timing → Risk Classification → Recommended Action → Human Review**

SyntaxError: invalid syntax (Temp/ipykernel_8640/2333478422.py, line 5)

# AI Manufacturing Shortage & Production Risk Agent

### AI-Assisted Manufacturing Decision-Support Prototype

**Version 1.0**

## Project Overview

This project demonstrates an AI-assisted, rule-based manufacturing decision-support prototype developed using Python, Pandas, Jupyter Notebook, and AI-assisted vibe coding.

The prototype analyzes manufacturing inventory, open purchase orders, production requirements, supplier lead times, and production timing to identify potential component shortages before they disrupt production.

## Business Objective

Provide production planners and purchasing professionals with earlier visibility into material shortages and supplier timing risks so they can evaluate corrective actions before production is affected.

## Decision Workflow

**Production Demand → Inventory → Open Purchase Orders → Shortage Detection → Supplier Lead Time → Production Timing → Risk Classification → Recommended Action → Human Review**

## Agent Capabilities

The AI Manufacturing Shortage & Production Risk Agent supports manufacturing decision-making by:

- Calculating net material availability
- Detecting component shortages
- Comparing inventory against production requirements
- Evaluating supplier lead times
- Comparing supplier lead time with the production schedule
- Classifying production risk as OK, LOW, MEDIUM, HIGH, or CRITICAL
- Generating recommended purchasing and production actions
- Keeping a human decision-maker responsible for final purchasing and supplier decisions

## Example Risk Analysis

The prototype identified the **P-103 Control Module** as a critical production risk:

- Required Quantity: 500 units
- Available Inventory: 200 units
- Open Purchase Order: 150 units
- Shortage: 150 units
- Supplier Lead Time: 30 days
- Production Requirement: 12 days
- Timing Gap: 18 days
- Risk Classification: **CRITICAL**

### Agent Recommendation

Evaluate purchase-order expediting, alternate suppliers, or production schedule adjustments.

**Human-in-the-Loop:** The agent provides decision support. Final purchasing, supplier, and production decisions remain with authorized personnel.

In [ ]:
import matplotlib.pyplot as plt

# Create a simple shortage risk chart
risk_data = df[df["Shortage_Qty"] > 0].copy()

plt.figure(figsize=(9, 5))

plt.bar(
    risk_data["Description"],
    risk_data["Shortage_Qty"]
)

plt.title("Manufacturing Component Shortage Risk")
plt.xlabel("Component")
plt.ylabel("Shortage Quantity (Units)")
plt.grid(axis="y", alpha=0.3)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import matplotlib.pyplot as plt

# Create a simple shortage risk chart
risk_data = df[df["Shortage_Qty"] > 0].copy()

In [ ]:
import pandas as pd

# Sample manufacturing inventory data
data = {

In [ ]:
import pandas as pd

# Sample manufacturing inventory data
data = {
    "Part_Number": ["P-101", "P-102", "P-103", "P-104"],
    "Description": [
        "Heating Element",
        "Temperature Sensor",
        "Control Module",
        "Wiring Harness"
    ],
    "Required_Qty": [500, 500, 500, 500],
    "Inventory_Qty": [320, 475, 200, 550],
    "Open_PO_Qty": [100, 100, 150, 0],
    "Lead_Time_Days": [21, 7, 30, 14],
    "Unit_Cost": [42, 18, 85, 26]
}

# Create dataframe
df = pd.DataFrame(data)

# Calculate available material
df["Net_Available"] = df["Inventory_Qty"] + df["Open_PO_Qty"]

# Calculate shortage
df["Shortage_Qty"] = (
    df["Required_Qty"] - df["Net_Available"]
).clip(lower=0)

df

In [ ]:
import matplotlib.pyplot as plt

# Select only components with shortages
risk_data = df[df["Shortage_Qty"] > 0].copy()

# Create shortage chart
plt.figure(figsize=(9, 5))

plt.bar(
    risk_data["Description"],
    risk_data["Shortage_Qty"]
)

plt.title("Manufacturing Component Shortage Risk")
plt.xlabel("Component")
plt.ylabel("Shortage Quantity (Units)")
plt.grid(axis="y", alpha=0.3)

plt.show()

In [ ]:
# Add production timing information
df["Days_Until_Production"] = [14, 20, 12, 30]

# Calculate supplier timing gap
df["Days_Late"] = (
    df["Lead_Time_Days"] - df["Days_Until_Production"]
).clip(lower=0)

# Classify production risk
def classify_risk(row):
    if row["Shortage_Qty"] == 0:
        return "OK"
    elif row["Days_Late"] >= 14:
        return "CRITICAL"
    elif row["Days_Late"] >= 7:
        return "HIGH"
    elif row["Days_Late"] > 0:
        return "MEDIUM"
    else:
        return "LOW"

df["Production_Risk"] = df.apply(classify_risk, axis=1)

# Display management risk summary
df[[
    "Part_Number",
    "Description",
    "Shortage_Qty",
    "Lead_Time_Days",
    "Days_Until_Production",
    "Days_Late",
    "Production_Risk"
]]

In [ ]:
# Executive Alert and Recommendation Engine

def executive_alert(row):

    if row["Production_Risk"] == "CRITICAL":
        return (
            f"IMMEDIATE ACTION: {row['Part_Number']} {row['Description']} "
            f"is {row['Shortage_Qty']} units short with "
            f"${row['Shortage_Cost_Exposure']:,.0f} in component exposure. "
            f"Supplier timing exceeds the production requirement by "
            f"{row['Days_Late']} days. "
            f"Evaluate PO expediting, alternate suppliers, "
            f"or production schedule adjustment."
        )

    elif row["Production_Risk"] == "HIGH":
        return (
            f"HIGH PRIORITY: {row['Part_Number']} {row['Description']} "
            f"is {row['Shortage_Qty']} units short with "
            f"${row['Shortage_Cost_Exposure']:,.0f} in component exposure. "
            f"Supplier timing gap is {row['Days_Late']} days. "
            f"Contact supplier and evaluate expedited delivery."
        )

    elif row["Production_Risk"] == "MEDIUM":
        return (
            f"MONITOR: {row['Part_Number']} {row['Description']} "
            f"has a potential production timing risk. "
            f"Confirm supplier delivery status."
        )

    elif row["Production_Risk"] == "LOW":
        return (
            f"MONITOR: {row['Part_Number']} {row['Description']} "
            f"has a material shortage, but current supplier timing "
            f"supports the production window."
        )

    else:
        return (
            f"OK: {row['Part_Number']} {row['Description']} "
            f"currently has sufficient material for production."
        )


# Apply the agent
df["Executive_Alert"] = df.apply(executive_alert, axis=1)

# Print executive alerts
for alert in df["Executive_Alert"]:
    print(alert)
    print()

In [ ]:
# Calculate financial exposure from component shortages
df["Shortage_Cost_Exposure"] = (
    df["Shortage_Qty"] * df["Unit_Cost"]
)

# Display financial risk summary
df[[
    "Part_Number",
    "Description",
    "Shortage_Qty",
    "Unit_Cost",
    "Shortage_Cost_Exposure",
    "Production_Risk"
]]

In [ ]:
%pip install pandas

# AI Manufacturing Shortage & Production Risk Agent

### AI-Assisted Manufacturing Decision-Support Prototype

This prototype analyzes manufacturing inventory, open purchase orders, supplier lead times, production requirements, and component costs to identify potential production shortages and supply-chain risks.

The agent calculates material availability, classifies production risk, estimates financial exposure, and generates management alerts and recommended actions.

**Human-in-the-Loop:** The system provides decision support. Final purchasing, supplier, and production decisions remain with authorized personnel.

## Project Objectives

The AI Manufacturing Shortage & Production Risk Agent is designed to:

- Identify component shortages before they affect production.
- Compare inventory and open purchase orders against production requirements.
- Evaluate supplier lead-time risk.
- Estimate financial exposure associated with component shortages.
- Classify production risk as **OK, LOW, MEDIUM, HIGH, or CRITICAL**.
- Generate management alerts and recommended actions.
- Support human decision-making rather than automatically making purchasing or production decisions.

## How the Agent Works

The prototype follows a five-step decision-support workflow:

1. **Collect Manufacturing Data** — Uses component requirements, current inventory, open purchase orders, supplier lead times, production timing, and unit costs.

2. **Calculate Material Availability** — Determines whether available inventory and incoming material can meet production requirements.

3. **Identify Shortages** — Calculates the quantity of material potentially missing for production.

4. **Assess Production Risk** — Evaluates shortages together with supplier lead times and production timing to classify risk.

5. **Generate Management Alerts** — Provides actionable alerts highlighting critical components, financial exposure, supplier timing concerns, and potential mitigation actions.

## System Architecture

The prototype uses a simple decision-support architecture:

**Manufacturing Data**  
↓  
**Inventory & Purchase Order Analysis**  
↓  
**Shortage Detection**  
↓  
**Supplier Lead-Time Analysis**  
↓  
**Production Risk Classification**  
↓  
**Financial Exposure Analysis**  
↓  
**AI-Assisted Management Alerts**  
↓  
**Human Review & Decision**

### Technology Stack

- **Python** — analysis and decision logic
- **Pandas** — manufacturing data processing
- **Matplotlib** — shortage and risk visualization
- **Jupyter Notebook** — prototype development and testing
- **GitHub** — version control and portfolio documentation

In [ ]:
# Calculate total material available
df["Net_Available"] = df["Inventory_Qty"] + df["Open_PO_Qty"]

# Calculate shortage quantity
df["Shortage_Qty"] = df["Required_Qty"] - df["Net_Available"]

# Prevent negative shortages
df["Shortage_Qty"] = df["Shortage_Qty"].clip(lower=0)

# Display updated table
df

In [ ]:
def classify_risk(row):
    if row["Shortage_Qty"] == 0:
        return "OK"
    elif row["Shortage_Qty"] >= 100 and row["Lead_Time_Days"] >= 21:
        return "CRITICAL"
    elif row["Shortage_Qty"] >= 50:
        return "HIGH"
    else:
        return "MEDIUM"

df["Production_Risk"] = df.apply(classify_risk, axis=1)

df[["Part_Number", "Description", "Shortage_Qty",
    "Lead_Time_Days", "Production_Risk"]]

In [76]:
# Calculate financial exposure from each shortage
df["Shortage_Cost_Exposure"] = df["Shortage_Qty"] * df["Unit_Cost"]

# Display financial risk
df[["Part_Number", "Description", "Shortage_Qty",
    "Unit_Cost", "Shortage_Cost_Exposure",
    "Production_Risk"]]

,Part_Number,Description,Shortage_Qty,Unit_Cost,Shortage_Cost_Exposure,Production_Risk
0,P-101,Heating Element,80,42,3360,HIGH
1,P-102,Temperature Sensor,0,18,0,OK
2,P-103,Control Module,150,85,12750,CRITICAL
3,P-104,Wiring Harness,0,26,0,OK


In [77]:
def recommend_action(row):
    if row["Production_Risk"] == "CRITICAL":
        return "Escalate immediately; contact supplier and evaluate alternate sourcing."
    elif row["Production_Risk"] == "HIGH":
        return "Expedite purchase order and confirm supplier delivery date."
    elif row["Production_Risk"] == "MEDIUM":
        return "Monitor inventory and supplier lead time."
    else:
        return "No immediate action required."

df["Recommended_Action"] = df.apply(recommend_action, axis=1)

df[["Part_Number", "Description", "Production_Risk",
    "Shortage_Cost_Exposure", "Recommended_Action"]]

,Part_Number,Description,Production_Risk,Shortage_Cost_Exposure,Recommended_Action
0,P-101,Heating Element,HIGH,3360,Expedite purchase order and confirm supplier d...
1,P-102,Temperature Sensor,OK,0,No immediate action required.
2,P-103,Control Module,CRITICAL,12750,Escalate immediately; contact supplier and eva...
3,P-104,Wiring Harness,OK,0,No immediate action required.


In [78]:
# Create management alert dashboard
alerts = df[df["Production_Risk"].isin(["HIGH", "CRITICAL"])][
    ["Part_Number", "Description", "Shortage_Qty",
     "Lead_Time_Days", "Shortage_Cost_Exposure",
     "Production_Risk", "Recommended_Action"]
]

print("AI MANUFACTURING RISK ALERT")
print("---------------------------")
print(f"Components requiring attention: {len(alerts)}")
print(f"Total shortage cost exposure: ${alerts['Shortage_Cost_Exposure'].sum():,.2f}")

alerts

AI MANUFACTURING RISK ALERT
---------------------------
Components requiring attention: 2
Total shortage cost exposure: $16,110.00


,Part_Number,Description,Shortage_Qty,Lead_Time_Days,Shortage_Cost_Exposure,Production_Risk,Recommended_Action
0,P-101,Heating Element,80,21,3360,HIGH,Expedite purchase order and confirm supplier d...
2,P-103,Control Module,150,30,12750,CRITICAL,Escalate immediately; contact supplier and eva...
